In [1]:
from os import environ
environ.setdefault("HF_HOME", "/data/hf_models/")

'/data/hf_models/'

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer

/data/agent_design_pattern/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
model = AutoModelForCausalLM.from_pretrained("ibm-granite/granite-4.0-h-1b")
tokenizer = AutoTokenizer.from_pretrained("ibm-granite/granite-4.0-h-1b")

The fast path is not available because one of `(selective_state_update, causal_conv1d_fn, causal_conv1d_update)` is None. Falling back to the naive implementation. To install follow https://github.com/state-spaces/mamba/#installation and https://github.com/Dao-AILab/causal-conv1d


In [ ]:
def get_current_temperature(location: str, unit: str):
    """
    Get the current temperature at a location.

    Args:
        location: The location to get the temperature for, in the format "City, Country"
        unit: The unit to return the temperature in. (choices: ["celsius", "fahrenheit"])
    """
    return 22.  # A real function should probably actually get the temperature!

def get_current_wind_speed(location: str):
    """
    Get the current wind speed in km/h at a given location.

    Args:
        location: The location to get the wind speed for, in the format "City, Country"
    """
    return 6.  # A real function should probably actually get the wind speed!

tools = [get_current_temperature, get_current_wind_speed]

In [6]:
messages = [
  {"role": "system", "content": "You are a bot that responds to weather queries. You should reply with the unit used in the queried location."},
  {"role": "user", "content": "Hey, what's the temperature in Paris right now?"}
]

In [ ]:
input_tokens = tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=True, return_dict=True, return_tensors="pt")
outputs = model.generate(**input_tokens.to(model.device), max_new_tokens=128)
# print(tokenizer.decode(outputs[0][len(inputs["input_ids"][0]):]))
response = tokenizer.batch_decode(outputs[:, input_tokens.input_ids.shape[-1]:], skip_special_tokens=True)

In [22]:
import re
import json
from typing import List

def extract_json(input_str: str) -> List[dict]:
    pattern = r'<tool_call>(?s:.*?)<\/tool_call>'
    matches = re.findall(pattern, input_str)
    return [match.strip().replace("<tool_call>", "").replace("</tool_call>", "").replace("\n", "") for match in matches]

print(response)
# list(map(extract_json, response))
a = extract_json(response[0])
str(a[0])

['<tool_call>\n{"name": "get_current_temperature", "arguments": {"location": "Paris, France", "unit": "celsius"}}\n</tool_call>']


'{"name": "get_current_temperature", "arguments": {"location": "Paris, France", "unit": "celsius"}}'

# Test Arxiv

In [33]:
import arxiv

query = "voting LLM"

# Construct the default API client.
client = arxiv.Client()

# Search for the 10 most recent articles matching the keyword "quantum."
search = arxiv.Search(
  query = query,
  max_results = 10,
  sort_by = arxiv.SortCriterion.SubmittedDate
)

results = client.results(search)
for result in results:
    print(f"{', '.join([author.name for author in result.authors])}, {result.title}")
    print(f"{result.entry_id} {result.published}")
    print(f"{result.summary}")
    print("-" * 50)
    print()

Martin Lackner, Piotr Skowron, Multi-Winner Voting with Approval Preferences
http://arxiv.org/abs/2007.01795v5 2020-07-03 16:33:50+00:00
Multi-winner voting is the process of selecting a fixed-size set of representative candidates based on voters' preferences. It occurs in applications ranging from politics (parliamentary elections) to the design of modern computer applications (collaborative filtering, dynamic Q&A platforms, diversifying search results). All these applications share the problem of identifying a representative subset of alternatives -- and the study of multi-winner voting is the principled analysis of this task.
  This book provides a thorough and in-depth look at multi-winner voting based on approval preferences. One speaks of approval preferences if voters express their preferences by providing a set of candidates they approve. Approval preferences thus separate candidates in approved and disapproved ones, a simple, binary classification. The corresponding multi-winn

In [1]:
system_prompt = "You are a helpful assistant"
user_prompt = "hi!"

In [2]:
from llama_index.llms.ollama import Ollama

llm = Ollama(model="qwen3:4b-thinking-2507-q4_K_M", base_url="192.168.55.1:11434", thinking=False, request_timeout=400, context_window=32768)

In [3]:
from llama_index.core.llms import ChatMessage
messages = [
    ChatMessage(role="system", content=system_prompt),
    ChatMessage(role="user", content=user_prompt)
]


In [4]:
response = llm.chat(messages)

In [5]:
print(response)

assistant: Okay, the user said "hi!" so I need to respond politely and friendly. Let me think about the best way to start.

First, I should greet them back. Since they used "hi!" which is informal, I can match that tone. Maybe say something like "Hello! How can I help you today?" That's friendly and opens the door for them to ask questions or need assistance.

Wait, the user might be testing if I'm working, so I should make sure to be clear and helpful. Let me check if there's any specific context I need to consider. But since they just said "hi!", probably no context. Just a standard greeting.

I should avoid any technical jargon here. Keep it simple. Maybe add a smiley emoji to make it warmer? Like a simple smiley at the end. But not too many. Let me see... "Hello! 😊 How can I help you today?"

Yes, that sounds good. Let me make sure it's not too long. Hmm, the user might be in a hurry, so short and sweet is better. Alright, I'll go with that.
</think>

Hello! 😊 How can I help you to

In [8]:
response.message.content

'Okay, the user said "hi!" so I need to respond politely and friendly. Let me think about the best way to start.\n\nFirst, I should greet them back. Since they used "hi!" which is informal, I can match that tone. Maybe say something like "Hello! How can I help you today?" That\'s friendly and opens the door for them to ask questions or need assistance.\n\nWait, the user might be testing if I\'m working, so I should make sure to be clear and helpful. Let me check if there\'s any specific context I need to consider. But since they just said "hi!", probably no context. Just a standard greeting.\n\nI should avoid any technical jargon here. Keep it simple. Maybe add a smiley emoji to make it warmer? Like a simple smiley at the end. But not too many. Let me see... "Hello! 😊 How can I help you today?"\n\nYes, that sounds good. Let me make sure it\'s not too long. Hmm, the user might be in a hurry, so short and sweet is better. Alright, I\'ll go with that.\n</think>\n\nHello! 😊 How can I help 

In [ ]:
# from a2a.types import AgentCapabilities, AgentCard, AgentSkill
from aap_core.types import AgentMessage
# from aap_core.orchestration import ReflectionAgent
# from aap_langchain.chain import ChatCausalMultiTurnsChain
# from langchain_ollama import ChatOllama
# from langchain_core.tools import tool
from aap_llamaindex.chain import ChatCausalMultiTurnsChain
from llama_index.llms.ollama import Ollama

In [2]:
# model_task = ChatOllama(model="llama3.2:latest", base_url="192.168.55.1:11434")
model_task = Ollama(model="qwen3:4b-thinking-2507-q4_K_M", base_url="192.168.55.1:11434", request_timeout=400, context_window=32768)


In [3]:
from typing import List

def get_weather(location: str, unit: str = "celsius") -> float:
    """Get the current temperature at a location.

    Args:
        location (str): The location to get the temperature for, in the format "City, Country"
        unit (str): The unit to return the temperature in. (choices: ["celsius", "fahrenheit"])

    Returns:
        The current temperature in the specified unit
    """
    return 22

def validate_user(user_id: int, addresses: List[str]) -> bool:
    """Validate user using historical addresses.

    Args:
        user_id (int): the user ID.
        addresses (List[str]): Previous addresses as a list of strings.
    """
    return True


In [4]:
system_prompt = "You are a helpful assistant. Only use tool when necessary"
user_prompt = "{query}"
task_chain = ChatCausalMultiTurnsChain(
    model=model_task,
    system_prompt=system_prompt,
    user_prompt_template=user_prompt,
    tools=[get_weather, validate_user],
    base_url="192.168.55.1:11434",
    request_timeout=400,
    context_window=32768)


In [5]:
query = "What is the current temperature in Paris, France?"
task_chain.invoke(AgentMessage(query=query))

[ChatMessage(role=<MessageRole.SYSTEM: 'system'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='You are a helpful assistant. Only use tool when necessary')]), ChatMessage(role=<MessageRole.USER: 'user'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='What is the current temperature in Paris, France?')]), ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text=''), ToolCallBlock(block_type='tool_call', tool_call_id=None, tool_name='get_weather', tool_kwargs={'location': 'Paris, France'})]), ChatMessage(role=<MessageRole.TOOL: 'tool'>, additional_kwargs={'tool_call_id': 'get_weather', 'name': 'get_weather'}, blocks=[TextBlock(block_type='text', text='22')]), ChatMessage(role=<MessageRole.ASSISTANT: 'assistant'>, additional_kwargs={}, blocks=[TextBlock(block_type='text', text='\n\nThe current temperature in Paris, France is **22°C**.')])]


AgentMessage(query='What is the current temperature in Paris, France?', query_media=None, query_media_type=None, origin='chain', responses=[('chain', '\n\nThe current temperature in Paris, France is **22°C**.')], context=None, execution_result='success', error_message=None, media=None, media_type=None)

In [ ]:
from pydantic import BaseModel
from typing import List

class Model1(BaseModel):
    name: List
    age: int



NameError: name 'List' is not defined

In [45]:
# a = Model1(name="John", age=30)
a = Model1.model_fields["name"].annotation
print(a)
print(issubclass(a, float))

<class 'str'>
False


In [ ]:
from typing import Callable, ParamSpec, TypeVar, Concatenate
# For Python < 3.10, use: from typing_extensions import Concatenate, ParamSpec

P = ParamSpec("P")
T = TypeVar("T")

# Define the type for the decorator's output function
# It prepends a 'str' argument to the original function's parameters (P)
DecoratorCallable = Callable[Concatenate[str, P], T]

def my_decorator(func: Callable[P, T]) -> DecoratorCallable[P, T]:
    def wrapper(new_arg: str, *args: P.args, **kwargs: P.kwargs) -> T:
        # new_arg is the prepended argument
        print(f"Prepended argument: {new_arg}")
        return func(*args, **kwargs)
    return wrapper

@my_decorator
def example_function(a: int, b: float) -> str:
    return f"a: {a}, b: {b}"

# When calling the decorated function, type checkers like mypy will expect
# the prepended 'str' argument first, followed by the original arguments.
result = example_function("hello", 1, 2.5)
print(result)


Prepended argument: hello
a: 1, b: 2.5


In [ ]:
from aap_dspy.chain import ChatCausalMultiTurnsChain
import dspy
from aap_core.types import AgentMessage

TypeError: The last parameter to Concatenate should be a ParamSpec variable.

In [2]:
lm = dspy.LM("ollama_chat/deepseek-r1:8b", api_base="http://192.168.55.1:11434", api_key="")
dspy.configure(lm=lm)

In [3]:
class QA(dspy.Signature):
    question: str = dspy.InputField()
    answer: str = dspy.OutputField()


predictor = dspy.Predict(QA)

In [4]:
from aap_core.types import AgentResponse


def msg2sig(message: AgentMessage) -> QA:
    return QA(question=message.query, answer="")


def sig2msg(signature: QA, name: str) -> AgentResponse:
    return (name, signature.answer)

In [5]:
chain = ChatCausalMultiTurnsChain(
    predictor=predictor,
    msg2sig=msg2sig,
    sig2msg=sig2msg,
    signature=QA
)

In [7]:
chain.invoke(AgentMessage(query="What is the current temperature in South Sudan?"))

AgentMessage(query='What is the current temperature in South Sudan?', query_media=None, origin='chain', responses=[('chain', 'I do not have access to real-time weather data or location-specific temperature information. To find the most accurate current temperature in South Sudan, please consult a reliable weather forecasting service or news outlet.')], context=None, execution_result='success', error_message=None, media=None)

In [26]:
QA(question="abc", answer="def")

QA(question='abc', answer='def')

In [ ]:
import numpy as np
from typing import List

def max_distance_subarray(S: np.ndarray, k: int) -> List[int]:
    """
    Given an number array S, an integer k. Write a function to get a subarray contain k elements,
    that the total distance formed by all pairs in the subarray is maximum
    """
    n = len(S)
    dp = np.zeros((n, k + 1), dtype=int)
    dp[:, 0] = 0
    for i in range(1, n):
        dp[i, 0] = dp[i - 1, 0] + np.abs(S[i] - S[i - 1])
    for j in range(1, k + 1):
        for i in range(j, n):
            dp[i, j] = np.maximum(
                dp[i - 1, j - 1] + np.abs(S[i] - S[i - j]), dp[i - 1, j]
            )
    res = []
    i, j = n - 1, k
    while j >= 0:
        if dp[i, j] == dp[i - 1, j]:
            i -= 1
        elif dp[i, j] == dp[i - 1, j - 1] + np.abs(S[i] - S[i - j]):
            res.append(S[i])
            i -= 1
            j -= 1
        else:
            i -= 1
    return res[::-1]

import math

def euclidean_distance(v1, v2):
    """Example distance function: Euclidean distance."""
    return math.sqrt(sum((a - b) ** 2 for a, b in zip(v1, v2)))

def max_vector_distance_subarray(S, k, dist_func=euclidean_distance):
    if k > len(S):
        return "k cannot be greater than the length of S."
    if k < 2:
        return S[:k]

    # 1. Find the two points that are furthest apart to start our set
    max_dist = -1
    best_pair = (0, 1)

    for i in range(len(S)):
        for j in range(i + 1, len(S)):
            d = dist_func(S[i], S[j])
            if d > max_dist:
                max_dist = d
                best_pair = (i, j)

    selected_indices = [best_pair[0], best_pair[1]]

    # 2. Greedily add points until we reach size k
    while len(selected_indices) < k:
        best_next_idx = -1
        max_total_extra_dist = -1

        for i in range(len(S)):
            if i in selected_indices:
                continue

            # Calculate how much this point adds to the total pairwise distance
            current_extra_dist = sum(dist_func(S[i], S[idx]) for idx in selected_indices)

            if current_extra_dist > max_total_extra_dist:
                max_total_extra_dist = current_extra_dist
                best_next_idx = i

        selected_indices.append(best_next_idx)

    return [S[idx] for idx in selected_indices]


In [13]:
# S = [1,5,10,153, 315, 143]
# S = [1, 10, 4, 12, 2, 25]
# k = 5
vectors = [(0, 0), (1, 1), (10, 10), (10, 0), (0, 10), (5, 5)]
k = 3
print(f"Selected Vectors: {max_vector_distance_subarray(vectors, k)}")

Selected Vectors: [(0, 0), (10, 10), (10, 0)]
